Scripts traducidos a Python 3


Este notebook contiene todos los scripts del repositorio del profe pasados a python 3

nombre: valeria valladares cámara

## Índice
1. [sampleScript.py](#1.-sampleScript)
2. [tcpclient.py](#2.-TCP-Client)
3. [udpclient.py](#3.-UDP-Client)
4. [tcpserver.py](#4.-TCP-Server)
5. [tcpproxy.py](#5.-TCP-Proxy)
6. [netcat.py](#6.-Netcat-en-Python)

---
## 1. sampleScript

Script de ejemplo. Ya era compatible con Python 3 (usa `print()` con paréntesis).

In [ ]:
# sampleScript.py: sin cambios porq ya era Python 3
print("Hello world!")

---
## 2. TCP cliente

**Cambios:**
 - client.send("") → client.send(b""): los sockets requieren bytes, no str.
 - print response → print(response.decode()): decodificar bytes a string para imprimir.



In [ ]:
import socket

target_host = "www.google.com"
target_port = 80

# Crear objeto socket
client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

# Conectar el cliente
client.connect((target_host, target_port))

# Enviar datos — Python 3: strings deben ser bytes (prefijo b"")
client.send(b"GET / HTTP/1.1\r\nHost: google.com\r\n\r\n")

# Recibir datos
response = client.recv(4096)

# Python 3: recv() devuelve bytes, decodificar para imprimir como texto
print(response.decode(errors='replace'))

client.close()

---
## 3. UDP Client

**Cambios Python:**
- client.sendto("AAABBBCCC", ...) → client.sendto(b"AAABBBCCC", ...): bytes requeridos.
- print data → print(data.decode()): decodificar bytes a string.



In [ ]:
import socket

target_host = "127.0.0.1"
target_port = 80

# Crear socket UDP (SOCK_DGRAM)
client = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)

# Enviar datos — Python 3: bytes con prefijo b""
client.sendto(b"AAABBBCCC", (target_host, target_port))

# Recibir datos
data, addr = client.recvfrom(4096)

# Python 3: decodificar bytes
print(data.decode(errors='replace'))

client.close()

---
## 4. TCP Server

**Cambios Python:**
- socket.socekt(...) → socket.socket(...): corrección de typo del original.
- print "..." % (...) → print(f"..." ): f-strings de Python 3.
- "$s:%d" → "%s:%d": corrección de typo ($ → %).
- client_socket.send("ACK!") → client_socket.send(b"ACK!") — bytes requeridos.
- request.decode() para imprimir el mensaje recibido.



In [ ]:
import socket
import threading

bind_ip   = "0.0.0.0"
bind_port = 9999

# Python 3: corregido typo "socekt" → "socket"
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)  # evitar error "address already in use"

server.bind((bind_ip, bind_port))
server.listen(5)

# Python 3: print() con paréntesis; corregido "$s" → "%s"
print("[*] Listening on %s:%d" % (bind_ip, bind_port))


def handle_client(client_socket):
    """Thread que maneja cada cliente conectado."""
    request = client_socket.recv(1024)

    # Python 3: recv() devuelve bytes → decodificar para imprimir
    print("[*] Received: %s" % request.decode(errors='replace'))

    # Python 3: send() requiere bytes → prefijo b""
    client_socket.send(b"ACK!")
    client_socket.close()


# Nota: la siguiente sección quedará bloqueada esperando conexiones.
# Comentar o adaptar según el entorno de uso.
"""
while True:
    client, addr = server.accept()
    # Python 3: corregido "$s:$d" → "%s:%d"
    print("[*] Accepted connection from: %s:%d" % (addr[0], addr[1]))

    client_handler = threading.Thread(target=handle_client, args=(client,))
    client_handler.start()
"""

print("[*] Servidor TCP listo. Descomenta el bucle 'while True' para aceptar conexiones.")

---
## 5. TCP Proxy

**Cambios:**
- print "..." → print("...") en todas las funciones.
- "$s:$d" → "%s:%d": corrección de typos en el original.
- buffer = "" → buffer = b"" acumuladores de red deben ser bytes.
- buffer += data funciona porque recv() ya devuelve bytes.
- isinstance(src, unicode) → isinstance(src, str): unicode` no existe en Python 3.
- xrange(...) → range(...): xrange fue eliminado en Python 3.
- ord(x) → simplificado con x directamente (en Python 3 iterar bytes da enteros).
- Indentación del original corregida (había errores de tabs).



In [ ]:
import sys
import socket
import threading


def hexdump(src, length=16):
    """Volcado hexadecimal del buffer recibido."""
    result = []

    # Python 3: 'unicode' no existe → usar 'str'; bytes es el tipo binario
    if isinstance(src, str):
        src = src.encode()

    # Python 3: xrange() → range()
    for i in range(0, len(src), length):
        s = src[i:i + length]
        # Python 3: iterar bytes devuelve int directamente, sin necesidad de ord()
        hexa = b' '.join(["%02X" % x for x in s])
        text = bytes([x if 0x20 <= x < 0x7F else ord('.') for x in s])
        result.append(b"%04X   %-*s   %s" % (i, length * 3, hexa, text))

    print(b'\n'.join(result).decode())


def receive_from(connection):
    """Lee todos los datos disponibles de un socket con timeout de 2 segundos."""
    # Python 3: acumulador debe ser bytes
    buffer = b""
    connection.settimeout(2)

    try:
        while True:
            data = connection.recv(4096)
            if not data:
                break
            buffer += data
    except Exception:
        pass

    return buffer


def request_handler(buffer):
    """Modificar paquetes hacia el host remoto (personalizable)."""
    return buffer


def response_handler(buffer):
    """Modificar respuestas hacia el cliente local (personalizable)."""
    return buffer


def proxy_handler(client_socket, remote_host, remote_port, receive_first):
    """Maneja el túnel entre cliente local y servidor remoto."""
    remote_socket = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    remote_socket.connect((remote_host, remote_port))

    if receive_first:
        remote_buffer = receive_from(remote_socket)
        hexdump(remote_buffer)
        remote_buffer = response_handler(remote_buffer)

        if len(remote_buffer):
            print("[<==] Sending %d bytes to localhost." % len(remote_buffer))
            client_socket.send(remote_buffer)

    while True:
        local_buffer = receive_from(client_socket)

        if len(local_buffer):
            print("[==>] Received %d bytes from localhost." % len(local_buffer))
            hexdump(local_buffer)
            local_buffer = request_handler(local_buffer)
            remote_socket.send(local_buffer)
            print("[==>] Sent to remote.")

            remote_buffer = receive_from(remote_socket)

            if len(remote_buffer):
                print("[<==] Received %d bytes from remote." % len(remote_buffer))
                hexdump(remote_buffer)
                remote_buffer = response_handler(remote_buffer)
                client_socket.send(remote_buffer)
                print("[<==] Sent to localhost.")

        if not len(local_buffer) or not len(remote_buffer):
            client_socket.close()
            remote_socket.close()
            print("[*] No more data. Closing connections.")
            break


def server_loop(local_host, local_port, remote_host, remote_port, receive_first):
    """Bucle principal del proxy: escucha conexiones locales."""
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    try:
        server.bind((local_host, local_port))
    except Exception:
        # Python 3: corregido "$s:$d" → "%s:%d"
        print("[!!] Failed to listen on %s:%d" % (local_host, local_port))
        print("[!!] Check for other listening sockets or correct permissions.")
        sys.exit(0)

    print("[*] Listening on %s:%d" % (local_host, local_port))
    server.listen(5)

    while True:
        client_socket, addr = server.accept()
        print("[==>] Received incoming connection from %s:%d" % (addr[0], addr[1]))

        proxy_thread = threading.Thread(
            target=proxy_handler,
            args=(client_socket, remote_host, remote_port, receive_first)
        )
        proxy_thread.start()


def tcp_proxy_main(args):
    """Punto de entrada del proxy. Reemplaza sys.argv para uso en Jupyter."""
    if len(args) != 5:
        print("Usage: tcp_proxy_main(['localhost', 'localport', 'remotehost', 'remoteport', 'receive_first'])")
        print("Example: tcp_proxy_main(['127.0.0.1', '9000', '10.12.132.1', '9000', 'True'])")
        return

    local_host   = args[0]
    local_port   = int(args[1])
    remote_host  = args[2]
    remote_port  = int(args[3])
    receive_first = args[4] == "True"

    server_loop(local_host, local_port, remote_host, remote_port, receive_first)


# Ejemplo de uso (comentado para no bloquear el kernel):
# tcp_proxy_main(['127.0.0.1', '9000', '10.12.132.1', '9000', 'True'])
print("[*] TCP Proxy definido. Llama a tcp_proxy_main([...]) para ejecutar.")

---
## 6. Netcat en Python

**Cambios:**
- print "..." → print("...") en todas las funciones (usage()).
- print str(err) → print(str(err)).
- raw_input("") → input(""): raw_input fue renombrado en Python 3.
- print response, → print(response.decode(errors='replace'), end=''): bytes + suprimir salto de línea.
- client.send(buffer) → client.send(buffer.encode()): convertir str a bytes.
- client,recv(4096) → client.recv(4096): corregido typo (coma → punto).
- client.socket, addr → client_socket, addr: corregido typo en server_loop.
- file_buffer como bytes y file_descriptor.write(file_buffer) sin cambios (ya bytes).
- client_socket.send(...) → encode cuando se envían strings.
- Variables globales mantenidas por compatibilidad con el diseño original.



In [ ]:
import sys
import socket
import getopt
import threading
import subprocess

# Variables globales
listen             = False
command            = False
upload             = False
execute            = ""
target             = ""
upload_destination = ""
port               = 0


def usage():
    # Python 3: print() con paréntesis
    print("BHP Net Tool")
    print()
    print("Usage: bhpnet.py -t target_host -p port")
    print("-l --listen                  - listen on [host]:[port] for incoming connections")
    print("-e --execute=file_to_run     - execute the given file upon receiving a connection")
    print("-c --command                 - initialize a command shell")
    print("-u --upload=destination      - upon receiving connection upload a file and write to [destination]")
    print()
    print("Examples:")
    print("bhpnet.py -t 192.168.0.1 -p 5555 -l -c")
    print("bhpnet.py -t 192.168.0.1 -p 5555 -l -u=c:\\\\target.exe")
    print('bhpnet.py -t 192.168.0.1 -p 5555 -l -e=\"cat /etc/passwd\"')
    print("echo 'ABCDEFGHI' | ./bhpnet.py -t 192.168.11.12 -p 135")
    sys.exit(0)


def run_command(cmd):
    """Ejecuta un comando del sistema y retorna su salida."""
    cmd = cmd.rstrip()
    try:
        output = subprocess.check_output(cmd, stderr=subprocess.STDOUT, shell=True)
    except Exception:
        output = b"Failed to execute command.\r\n"
    return output


def client_sender(buffer):
    """Envía datos al servidor y mantiene una sesión interactiva."""
    client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)

    try:
        client.connect((target, port))

        if len(buffer):
            # Python 3: encode str → bytes para send()
            client.send(buffer.encode())

        while True:
            recv_len = 1
            response = b""

            while recv_len:
                # Python 3: corregido typo "client,recv" → "client.recv"
                data     = client.recv(4096)
                recv_len = len(data)
                response += data

                if recv_len < 4096:
                    break

            # Python 3: decodificar bytes; end='' suprime salto de línea extra
            print(response.decode(errors='replace'), end='')

            # Python 3: raw_input() → input()
            buffer  = input("")
            buffer += "\n"

            client.send(buffer.encode())

    except Exception:
        print("[*] Exception! Exiting.")
        client.close()


def client_handler(client_socket):
    """Maneja upload de archivos, ejecución de comandos y shell interactivo."""
    global upload_destination, execute, command

    # Manejo de upload
    if len(upload_destination):
        file_buffer = b""  # Python 3: bytes

        while True:
            data = client_socket.recv(1024)
            if not data:
                break
            file_buffer += data

        try:
            with open(upload_destination, "wb") as fd:
                fd.write(file_buffer)
            msg = "Successfully saved file to %s\r\n" % upload_destination
            client_socket.send(msg.encode())  # Python 3: encode
        except Exception:
            msg = "Failed to save file to %s\r\n" % upload_destination
            client_socket.send(msg.encode())

    # Ejecución de comando
    if len(execute):
        output = run_command(execute)
        client_socket.send(output)  # ya es bytes desde run_command()

    # Shell interactivo
    if command:
        while True:
            client_socket.send(b"<BHP:#> ")  # Python 3: bytes

            cmd_buffer = ""
            while "\n" not in cmd_buffer:
                # Python 3: decode bytes a str para acumular
                cmd_buffer += client_socket.recv(1024).decode(errors='replace')

            response = run_command(cmd_buffer)
            client_socket.send(response)


def server_loop():
    """Escucha conexiones entrantes y lanza un thread por cliente."""
    global target

    if not len(target):
        target = "0.0.0.0"

    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    server.bind((target, port))
    server.listen(5)

    while True:
        # Python 3: corregido "client.socket" → "client_socket"
        client_socket, addr = server.accept()
        client_thread = threading.Thread(target=client_handler, args=(client_socket,))
        client_thread.start()


def bhp_main(argv=None):
    """Punto de entrada. Acepta lista de args o usa sys.argv."""
    global listen, port, execute, command, upload_destination, target

    if argv is None:
        argv = sys.argv[1:]

    if not len(argv):
        usage()

    try:
        opts, args = getopt.getopt(
            argv, "hle:t:p:cu:",
            ["help", "listen", "execute=", "target=", "port=", "command", "upload="]
        )
    except getopt.GetoptError as err:
        print(str(err))  # Python 3: print()
        usage()

    for o, a in opts:
        if o in ("-h", "--help"):
            usage()
        elif o in ("-l", "--listen"):
            listen = True
        elif o in ("-e", "--execute"):
            execute = a
        elif o in ("-c", "--command"):
            command = True
        elif o in ("-u", "--upload"):
            upload_destination = a
        elif o in ("-t", "--target"):
            target = a
        elif o in ("-p", "--port"):
            port = int(a)
        else:
            assert False, "Unhandled option"

    if not listen and len(target) and port > 0:
        buffer = sys.stdin.read()
        client_sender(buffer)

    if listen:
        server_loop()


# Para ejecutar desde Jupyter, pasar args como lista:
# bhp_main(["-t", "192.168.0.1", "-p", "5555", "-l", "-c"])
print("[*] BHP Netcat (Python 3) definido. Llama a bhp_main([...]) para ejecutar.")

---
## Resumen de cambios Python 2 → Python 3

| Patrón Python 2 | Equivalente Python 3 | Motivo |
|---|---|---|
| `print x` | `print(x)` | `print` es función en Py3 |
| `raw_input()` | `input()` | Renombrado en Py3 |
| `socket.send("str")` | `socket.send(b"str")` | Sockets requieren bytes |
| `socket.recv()` → str | `socket.recv()` → bytes | Requiere `.decode()` |
| `xrange()` | `range()` | `xrange` eliminado en Py3 |
| `isinstance(x, unicode)` | `isinstance(x, str)` | `unicode` eliminado en Py3 |
| `ord(byte)` en bytes loop | `byte` directamente | Iterar bytes da `int` en Py3 |
| `buffer = ""` (red) | `buffer = b""` | Acumuladores de red son bytes |